# APL Logistics: Customer, Product & Profitability Performance
## Run each numbered code cell in order
This notebook follows the supplied brief. Monetary amounts are **dataset currency units**, not assumed USD. All statuses are included in the primary analysis; a COMPLETE/CLOSED sensitivity view is separate. Record-level economics are not verified recognized revenue. No dates or order IDs are available, so no time trends or distinct-order counts are fabricated.

**Execution:** unzip the project, place the supplied CSV in the project directory or its parent, and open this notebook in Jupyter. The commented Python companion contains the same numbered cells.

### Cell 01 — Install dependencies (run once)

In [1]:
%pip install -r requirements.txt

Analysis dependencies installed; pip installation skipped during preparation.


### Cell 02 — Import libraries and configure display

In [2]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
OUT = Path("outputs")
OUT.mkdir(exist_ok=True)

### Cell 03 — Define all validation and analysis functions

In [3]:
"""Shared, auditable calculations for the notebook and dashboard."""
from pathlib import Path
import numpy as np
import pandas as pd

TOLERANCE = 0.02
NUMERIC = ['Sales', 'Order Item Total', 'Order Profit Per Order', 'Benefit per order',
           'Sales per customer', 'Order Item Discount', 'Order Item Discount Rate',
           'Order Item Product Price', 'Order Item Quantity', 'Order Item Profit Ratio',
           'Customer Id', 'Order Customer Id']
GROUPS = ['Customer Segment', 'Category Name', 'Product Name', 'Market',
          'Order Region', 'Order Country', 'Order Status']

def load_csv(source):
    """Read a local path or Streamlit upload without silently replacing bad bytes."""
    try:
        return pd.read_csv(source, encoding='utf-8', dtype={'Customer Zipcode': 'string'})
    except UnicodeDecodeError:
        if hasattr(source, 'seek'):
            source.seek(0)
        return pd.read_csv(source, encoding='cp1252', dtype={'Customer Zipcode': 'string'})

def clean_data(raw):
    """Keep losses and incomplete statuses; quarantine invalid financial records."""
    d = raw.copy()
    required = set(NUMERIC + GROUPS)
    missing = required - set(d.columns)
    if missing:
        raise ValueError(f'Missing required columns: {sorted(missing)}')
    for column in d.select_dtypes(include=['object', 'string']).columns:
        d[column] = d[column].astype('string').str.strip().replace('', pd.NA)
    for column in NUMERIC:
        d[column] = pd.to_numeric(d[column], errors='coerce')
    for column in GROUPS:
        d[column] = d[column].fillna('Unknown')
    checks = pd.DataFrame(index=d.index)
    checks['missing_or_nonfinite_numeric'] = ~np.isfinite(d[NUMERIC].to_numpy(dtype=float)).all(axis=1)
    checks['nonpositive_sales'] = d['Sales'].le(0) | d['Order Item Total'].le(0)
    checks['invalid_quantity'] = d['Order Item Quantity'].le(0) | d['Order Item Quantity'].mod(1).ne(0)
    checks['invalid_discount'] = ~d['Order Item Discount Rate'].between(0, 1) | d['Order Item Discount'].lt(0)
    checks['gross_net_mismatch'] = (d['Sales'] - d['Order Item Discount'] - d['Order Item Total']).abs().gt(TOLERANCE)
    checks['price_quantity_mismatch'] = (d['Sales'] - d['Order Item Product Price'] * d['Order Item Quantity']).abs().gt(TOLERANCE)
    checks['profit_alias_mismatch'] = (d['Benefit per order'] - d['Order Profit Per Order']).abs().gt(TOLERANCE)
    checks['net_sales_alias_mismatch'] = (d['Sales per customer'] - d['Order Item Total']).abs().gt(TOLERANCE)
    checks['customer_id_mismatch'] = d['Customer Id'].ne(d['Order Customer Id'])
    invalid = checks.any(axis=1)
    # Do not remove identical rows: without a line ID they could be valid repeated items.
    audit = pd.DataFrame({'issue': checks.columns, 'rows_flagged': checks.sum().values})
    rejected = d.loc[invalid].copy()
    rejected['rejection_reason'] = [
        '; '.join(row.index[row]) for _, row in checks.loc[invalid].iterrows()
    ]
    d = d.loc[~invalid].copy()
    d['gross_sales'] = d['Sales']
    d['net_revenue'] = d['Order Item Total']
    d['profit'] = d['Order Profit Per Order']
    d['discount_amount'] = d['Order Item Discount']
    d['discount_rate'] = d['Order Item Discount Rate']
    d['margin_pct'] = 100 * d['profit'] / d['net_revenue']
    d['implied_cost'] = d['net_revenue'] - d['profit']
    d['discount_band'] = pd.cut(d['discount_rate'], bins=[-0.000001, 0, .05, .10, .15, .20, 1],
                               labels=['0%', '>0–5%', '>5–10%', '>10–15%', '>15–20%', '>20%'])
    d['Customer Id'] = d['Customer Id'].astype('int64')
    return d, audit, rejected

def summarize(d, keys):
    result = d.groupby(keys, observed=True, dropna=False).agg(
        rows=('profit', 'size'), customers=('Customer Id', 'nunique'),
        gross_sales=('gross_sales', 'sum'), net_revenue=('net_revenue', 'sum'),
        profit=('profit', 'sum'), discounts=('discount_amount', 'sum'),
        units=('Order Item Quantity', 'sum')).reset_index()
    result['margin_pct'] = 100 * result['profit'] / result['net_revenue']
    result['profit_to_gross_pct'] = 100 * result['profit'] / result['gross_sales']
    result['effective_discount_pct'] = 100 * result['discounts'] / result['gross_sales']
    return result.sort_values('profit', ascending=False)

def customer_table(d):
    c = summarize(d, ['Customer Id'])
    # Explicit project definition; not an externally standardized index.
    c['customer_value_index'] = c['profit']
    positive = c.loc[c['profit'] > 0, 'profit']
    cutoff = positive.quantile(.8) if len(positive) else np.inf
    c['value_tier'] = np.select([c['profit'].le(0), c['profit'].ge(cutoff)],
                                 ['Nonpositive profit', 'High value'], default='Positive core')
    c['low_margin'] = c['profit'].gt(0) & c['margin_pct'].lt(5)
    positive_total = c['profit'].clip(lower=0).sum()
    c['cumulative_positive_profit_pct'] = (100 * c['profit'].clip(lower=0).cumsum() / positive_total
                                           if positive_total else np.nan)
    return c

def kpis(d):
    gross, net, profit = d['gross_sales'].sum(), d['net_revenue'].sum(), d['profit'].sum()
    discount = d['discount_amount'].sum()
    # Fixed-cost, fixed-volume counterfactual: restoring discounts adds discount dollars to profit.
    baseline_profit = profit + discount
    return {'rows': len(d), 'customers': d['Customer Id'].nunique(),
            'gross_sales': gross, 'net_revenue': net, 'profit': profit,
            'discounts': discount, 'margin_pct': 100 * profit / net if net else np.nan,
            'profit_to_gross_pct': 100 * profit / gross if gross else np.nan,
            'discount_impact_ratio_pct': 100 * discount / baseline_profit if baseline_profit else np.nan,
            'modeled_margin_erosion_pp': 100 * baseline_profit / gross - 100 * profit / net if gross and net else np.nan}

def discount_scenario(d, new_rate):
    """Uniform replacement rate; costs/quantities unchanged, no demand response."""
    gross = d['gross_sales'].sum()
    modeled_net = gross * (1 - new_rate)
    modeled_profit = modeled_net - d['implied_cost'].sum()
    return pd.Series({'scenario_discount_pct': 100 * new_rate,
                      'scenario_net_revenue': modeled_net, 'scenario_profit': modeled_profit,
                      'profit_change': modeled_profit - d['profit'].sum(),
                      'scenario_margin_pct': 100 * modeled_profit / modeled_net if modeled_net else np.nan})

def resolve_data():
    candidates = [Path('APL_Logistics (1).csv'), Path('../APL_Logistics (1).csv')]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError('Put APL_Logistics (1).csv beside the notebook or one directory above it.')

### Cell 04 — Load the supplied CSV

In [4]:
DATA_PATH = resolve_data()
raw = load_csv(DATA_PATH)
print("Dataset:", DATA_PATH)
print("Rows and columns:", raw.shape)
display(raw.head().drop(columns=["Customer Fname", "Customer Lname", "Customer Street", "Customer Zipcode", "Latitude", "Longitude"]))

Dataset: ../APL_Logistics (1).csv
Rows and columns: (180519, 40)


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,Customer Country,Customer Id,Customer Segment,Customer State,Department Id,...,Order Customer Id,Order Item Discount,Order Item Discount Rate,Order Item Product Price,Order Item Profit Ratio,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Region,Order State,Order Status,Product Name,Product Price,Shipping Mode
0,DEBIT,6,4,159.690,472.450,Late delivery,1,9,Cardio Equipment,Brownsville,EE. UU.,1,Consumer,TX,3,...,1,27.500,0.060,99.990,0.340,5,499.950,472.450,159.690,South Asia,Maharashtra,COMPLETE,Nike Men's Free 5.0+ Running Shoe,99.990,Standard Class
1,DEBIT,4,4,48.710,167.960,Shipping on time,0,29,Shop By Sport,Littleton,EE. UU.,2,Consumer,CO,5,...,2,31.990,0.160,39.990,0.290,5,199.950,167.960,48.710,Central America,Cortés,ON_HOLD,Under Armour Girls' Toddler Spine Surge Runni,39.990,Standard Class
2,DEBIT,4,4,87.360,181.990,Shipping on time,0,48,Water Sports,Littleton,EE. UU.,2,Consumer,CO,7,...,2,18.000,0.090,199.990,0.480,1,199.990,181.990,87.360,Central America,Cortés,ON_HOLD,Pelican Sunstream 100 Kayak,199.990,Standard Class
3,DEBIT,6,4,-41.890,175.990,Late delivery,1,48,Water Sports,Littleton,EE. UU.,2,Consumer,CO,7,...,2,24.000,0.120,199.990,-0.240,1,199.990,175.990,-41.890,East of USA,Nueva York,COMPLETE,Pelican Sunstream 100 Kayak,199.990,Standard Class
4,DEBIT,6,4,10.000,40.000,Late delivery,1,24,Women's Apparel,Littleton,EE. UU.,2,Consumer,CO,5,...,2,10.000,0.200,50.000,0.250,1,50.000,40.000,10.000,East of USA,Nueva York,COMPLETE,Nike Men's Dri-FIT Victory Golf Polo,50.000,Standard Class


### Cell 05 — Inspect schema, missingness and duplicates

In [5]:
display(raw.dtypes.to_frame("dtype"))
missing = raw.isna().sum().sort_values(ascending=False).to_frame("missing_rows")
display(missing[missing["missing_rows"] > 0])
print("Exact duplicate rows:", raw.duplicated().sum())
print("Dates present:", [c for c in raw.columns if "date" in c.lower()])
print("Do not deduplicate without a verified line identifier.")

,dtype
Type,object
Days for shipping (real),int64
Days for shipment (scheduled),int64
Benefit per order,float64
Sales per customer,float64
Delivery Status,object
Late_delivery_risk,int64
Category Id,int64
Category Name,object
Customer City,object


,missing_rows
Customer Lname,8
Customer Zipcode,3


Exact duplicate rows: 0
Dates present: []
Do not deduplicate without a verified line identifier.


### Cell 06 — Clean and quarantine invalid financial records

In [6]:
df, audit, rejected = clean_data(raw)
display(audit)
print(f"Retained: {len(df):,}; quarantined: {len(rejected):,}")
audit.to_csv(OUT / "validation_audit.csv", index=False)
# Raw rejected data may contain PII; save only counts/reasons.
rejected["rejection_reason"].value_counts().to_csv(OUT / "rejection_reasons.csv")
assert len(df) + len(rejected) == len(raw)
assert np.allclose(df["gross_sales"] - df["discount_amount"], df["net_revenue"], atol=.02)
assert (df["net_revenue"] > 0).all()

,issue,rows_flagged
0,missing_or_nonfinite_numeric,0
1,nonpositive_sales,0
2,invalid_quantity,0
3,invalid_discount,0
4,gross_net_mismatch,0
5,price_quantity_mismatch,0
6,profit_alias_mismatch,0
7,net_sales_alias_mismatch,0
8,customer_id_mismatch,0


Retained: 180,519; quarantined: 0


**Definitions.** Gross revenue = sum of `Sales` (the brief's total-revenue KPI). Net revenue = sum of `Order Item Total`. Reported profit = sum of `Order Profit Per Order` once; `Benefit per order` duplicates it. Net margin = 100 × total profit / net revenue, not the simple average of item ratios. Customer Value Index (project definition) = aggregate reported profit per customer. Discount Impact Ratio = 100 × discounts / (reported profit + discounts), a fixed-cost/no-demand-response modeled profit reduction. Modeled margin erosion compares no-discount margin with observed net margin; neither establishes causality. Costs inferred as net revenue minus profit are not observed shipping costs.

### Cell 07 — Calculate core financial KPIs

In [7]:
metrics = kpis(df)
display(pd.Series(metrics, name="value").to_frame())
print("Aggregate rounding residual (gross - discounts - net):", metrics["gross_sales"] - metrics["discounts"] - metrics["net_revenue"])
(OUT / "kpis.json").write_text(json.dumps(metrics, indent=2, default=float))

,value
rows,"180,519.000"
customers,"20,652.000"
gross_sales,"36,784,734.310"
net_revenue,"33,054,402.040"
profit,"3,966,902.970"
discounts,"3,730,378.400"
margin_pct,12.001
profit_to_gross_pct,10.784
discount_impact_ratio_pct,48.464
modeled_margin_erosion_pp,8.924


Aggregate rounding residual (gross - discounts - net): -46.12999999523163


341

### Cell 08 — Visualize gross sales, discounts, net revenue and profit

In [8]:
fig, ax = plt.subplots(figsize=(10, 5))
labels = ["Gross sales", "Discounts", "Net revenue", "Reported profit"]
values = [metrics[k] for k in ["gross_sales", "discounts", "net_revenue", "profit"]]
sns.barplot(x=labels, y=values, ax=ax)
ax.set(ylabel="Dataset currency units", title="Revenue and profit overview (all statuses)")
fig.tight_layout()
fig.savefig(OUT / "financial_overview.png", dpi=160)
plt.show()
print("Time trends cannot be computed: no date column was supplied.")

Time trends cannot be computed: no date column was supplied.


### Cell 09 — Analyze order-status sensitivity

In [9]:
status_summary = summarize(df, ["Order Status"])
display(status_summary)
finalized_proxy = df[df["Order Status"].isin(["COMPLETE", "CLOSED"])]
display(pd.DataFrame({"all_statuses": kpis(df), "complete_closed_proxy": kpis(finalized_proxy)}))
print("COMPLETE/CLOSED is a sensitivity proxy, not a verified accounting recognition policy.")
status_summary.to_csv(OUT / "status_summary.csv", index=False)

,Order Status,rows,customers,gross_sales,net_revenue,profit,discounts,units,margin_pct,profit_to_gross_pct,effective_discount_pct
2,COMPLETE,59491,12523,"12,095,314.720","10,859,271.620","1,321,735.750","1,236,058.150",126624,12.171,10.928,10.219
6,PENDING_PAYMENT,39832,9747,"8,106,697.400","7,286,275.560","843,810.240","820,431.530",84736,11.581,10.409,10.120
7,PROCESSING,21902,6287,"4,504,063.660","4,051,380.730","494,825.870","452,688.600",46623,12.214,10.986,10.051
1,CLOSED,19616,5900,"4,022,624.090","3,617,466.810","457,981.090","405,162.180",41458,12.660,11.385,10.072
5,PENDING,20227,5887,"4,120,532.790","3,704,456.190","435,725.850","416,082.180",43196,11.762,10.575,10.098
3,ON_HOLD,9804,3232,"1,981,542.670","1,780,237.680","208,913.040","201,307.560",20845,11.735,10.543,10.159
8,SUSPECTED_FRAUD,4062,1429,"825,934.940","741,492.310","85,136.710","84,443.740",8613,11.482,10.308,10.224
0,CANCELED,3692,1320,"744,370.390","668,244.980","75,345.630","76,126.340",7875,11.275,10.122,10.227
4,PAYMENT_REVIEW,1893,687,"383,653.650","345,576.160","43,428.790","38,078.120",4109,12.567,11.320,9.925


,all_statuses,complete_closed_proxy
rows,"180,519.000","79,107.000"
customers,"20,652.000","14,543.000"
gross_sales,"36,784,734.310","16,117,938.810"
net_revenue,"33,054,402.040","14,476,738.430"
profit,"3,966,902.970","1,779,716.840"
discounts,"3,730,378.400","1,641,220.330"
margin_pct,12.001,12.294
profit_to_gross_pct,10.784,11.042
discount_impact_ratio_pct,48.464,47.976
modeled_margin_erosion_pp,8.924,8.931


COMPLETE/CLOSED is a sensitivity proxy, not a verified accounting recognition policy.


### Cell 10 — Aggregate product and category profitability

In [10]:
products = summarize(df, ["Product Name"])
categories = summarize(df, ["Category Name"])
display(products.head(15))
display(categories)
products.to_csv(OUT / "product_profitability.csv", index=False)
categories.to_csv(OUT / "category_profitability.csv", index=False)

,Product Name,rows,customers,gross_sales,net_revenue,profit,discounts,units,margin_pct,profit_to_gross_pct,effective_discount_pct
24,Field & Stream Sportsman 16 Gun Fire Safe,17325,8759,"6,929,653.500","6,226,935.120","756,220.760","702,728.000",17325,12.144,10.913,10.141
71,Perfect Fitness Perfect Rip Deck,24515,10046,"4,421,143.020","3,973,180.310","493,828.300","447,968.170",73698,12.429,11.170,10.132
21,Diamondback Women's Serene Classic Comfort Bi,13729,7788,"4,118,425.420","3,700,783.540","427,455.570","417,649.500",13729,11.550,10.379,10.141
61,Nike Men's Free 5.0+ Running Shoe,12169,7377,"3,667,633.200","3,295,693.320","379,915.820","371,942.590",36680,11.528,10.359,10.141
59,Nike Men's Dri-FIT Victory Golf Polo,21035,9468,"3,147,800.000","2,828,708.500","350,421.030","319,091.500",62956,12.388,11.132,10.137
70,Pelican Sunstream 100 Kayak,15500,8277,"3,099,845.000","2,785,518.000","324,076.370","314,327.000",15500,11.634,10.455,10.140
67,O'Brien Men's Neoprene Life Vest,19298,9134,"2,888,993.940","2,596,454.040","318,451.430","292,550.630",57803,12.265,11.023,10.126
56,Nike Men's CJ Elite 2 TD Football Cleat,22246,9717,"2,891,757.540","2,598,494.440","311,902.820","293,263.100",22246,12.003,10.786,10.141
102,Under Armour Girls' Toddler Spine Surge Runni,10617,6774,"1,269,082.650","1,140,770.950","126,278.510","128,314.030",31735,11.070,9.950,10.111
18,Dell Laptop,442,442,"663,000.000","595,395.000","69,656.810","67,605.000",442,11.699,10.506,10.197


,Category Name,rows,customers,gross_sales,net_revenue,profit,discounts,units,margin_pct,profit_to_gross_pct,effective_discount_pct
18,Fishing,17325,8759,"6,929,653.500","6,226,935.120","756,220.760","702,728.000",17325,12.144,10.913,10.141
12,Cleats,24551,10049,"4,431,942.660","3,982,856.450","494,636.920","449,091.670",73734,12.419,11.161,10.133
9,Camping & Hiking,13729,7788,"4,118,425.420","3,700,783.540","427,455.570","417,649.500",13729,11.550,10.379,10.141
10,Cardio Equipment,12487,7486,"3,694,843.200","3,320,250.720","383,011.100","374,595.190",37587,11.536,10.366,10.138
47,Women's Apparel,21035,9468,"3,147,800.000","2,828,708.500","350,421.030","319,091.500",62956,12.388,11.132,10.137
46,Water Sports,15540,8281,"3,113,844.600","2,798,044.100","325,146.960","315,800.500",15540,11.621,10.442,10.142
30,Indoor/Outdoor Games,19298,9134,"2,888,993.940","2,596,454.040","318,451.430","292,550.630",57803,12.265,11.023,10.126
34,Men's Footwear,22246,9717,"2,891,757.540","2,598,494.440","311,902.820","293,263.100",22246,12.003,10.786,10.141
38,Shop By Sport,10984,6935,"1,309,522.020","1,177,185.620","129,813.960","132,338.730",32726,11.027,9.913,10.106
13,Computers,442,442,"663,000.000","595,395.000","69,656.810","67,605.000",442,11.699,10.506,10.197


### Cell 11 — Find loss-makers and high-revenue/low-margin products

In [11]:
high_revenue_cutoff = products["net_revenue"].quantile(.75)
low_margin_cutoff = products["margin_pct"].quantile(.25)
priority_products = products[(products["net_revenue"] >= high_revenue_cutoff) & (products["margin_pct"] <= low_margin_cutoff)]
print("High revenue = top quartile; low margin = bottom quartile (product-level).")
print("Revenue cutoff:", high_revenue_cutoff, "Margin cutoff:", low_margin_cutoff)
display(priority_products)
print("Loss-making categories:")
display(categories[categories["profit"] < 0])
priority_products.to_csv(OUT / "priority_products.csv", index=False)

High revenue = top quartile; low margin = bottom quartile (product-level).
Revenue cutoff: 36597.1825 Margin cutoff: 9.750151269525585


,Product Name,rows,customers,gross_sales,net_revenue,profit,discounts,units,margin_pct,profit_to_gross_pct,effective_discount_pct
17,DVDs,483,483,"79,395.540","71,319.380","6,655.430","8,076.430",483,9.332,8.383,10.172
0,Adult dog supplies,492,492,"41,524.800","37,318.300","3,589.260","4,206.500",492,9.618,8.644,10.130
48,Men's gala suit,208,208,"43,856.800","39,365.710","2,006.040","4,491.210",208,5.096,4.574,10.241


Loss-making categories:


,Category Name,rows,customers,gross_sales,net_revenue,profit,discounts,units,margin_pct,profit_to_gross_pct,effective_discount_pct


### Cell 12 — Plot category-by-market weighted margin heatmap

In [12]:
category_market = summarize(df, ["Category Name", "Market"])
heat = category_market.pivot(index="Category Name", columns="Market", values="margin_pct")
fig, ax = plt.subplots(figsize=(11, 16))
sns.heatmap(heat, cmap="RdYlGn", center=0, ax=ax, cbar_kws={"label": "Net margin (%)"})
ax.set_title("Category × market weighted profit margin")
fig.tight_layout()
fig.savefig(OUT / "category_market_heatmap.png", dpi=160)
plt.show()

### Cell 13 — Calculate customer contribution and value tiers

In [13]:
customers = customer_table(df)
display(customers.head(10))
display(customers.nsmallest(10, "profit"))
tiers = customers.groupby("value_tier").agg(customers=("Customer Id", "size"), net_revenue=("net_revenue", "sum"), profit=("profit", "sum"))
tiers["margin_pct"] = 100 * tiers["profit"] / tiers["net_revenue"]
display(tiers)
customers.to_csv(OUT / "customer_profitability.csv", index=False)
tiers.to_csv(OUT / "customer_tiers.csv")

,Customer Id,rows,customers,gross_sales,net_revenue,profit,discounts,units,margin_pct,profit_to_gross_pct,effective_discount_pct,customer_value_index,value_tier,low_margin,cumulative_positive_profit_pct
2612,2641,43,1,"9,130.920","8,221.640","2,441.970",909.290,90,29.702,26.744,9.958,"2,441.970",High value,False,0.048
1636,1657,42,1,"9,223.710","8,165.230","2,196.920","1,058.490",111,26.906,23.818,11.476,"2,196.920",High value,False,0.092
9755,9833,24,1,"6,059.380","5,386.460","1,938.390",672.920,48,35.986,31.990,11.105,"1,938.390",High value,False,0.130
2597,2626,27,1,"6,274.360","5,477.840","1,928.570",796.540,55,35.207,30.737,12.695,"1,928.570",High value,False,0.169
4958,5004,45,1,"8,164.700","7,499.370","1,917.990",665.350,114,25.575,23.491,8.149,"1,917.990",High value,False,0.207
3697,3735,27,1,"6,019.330","5,532.410","1,906.360",486.920,61,34.458,31.671,8.089,"1,906.360",High value,False,0.244
741,749,38,1,"7,649.380","6,953.400","1,855.150",695.980,64,26.680,24.252,9.099,"1,855.150",High value,False,0.281
5509,5560,29,1,"6,528.210","6,118.590","1,831.460",409.620,74,29.933,28.055,6.275,"1,831.460",High value,False,0.317
10875,10967,27,1,"5,734.410","5,181.650","1,822.330",552.760,64,35.169,31.779,9.639,"1,822.330",High value,False,0.353
5007,5053,33,1,"7,411.320","6,696.680","1,813.340",714.640,67,27.078,24.467,9.643,"1,813.340",High value,False,0.389


,Customer Id,rows,customers,gross_sales,net_revenue,profit,discounts,units,margin_pct,profit_to_gross_pct,effective_discount_pct,customer_value_index,value_tier,low_margin,cumulative_positive_profit_pct
1410,1428,13,1,"3,883.670","3,631.040","-3,868.560",252.640,28,-106.541,-99.611,6.505,"-3,868.560",Nonpositive profit,False,100.000
13980,14086,1,1,"1,500.000","1,350.000","-3,442.500",150.000,1,-255.000,-229.500,10.000,"-3,442.500",Nonpositive profit,False,100.000
18003,18109,1,1,"1,500.000","1,320.000","-3,366.000",180.000,1,-255.000,-224.400,12.000,"-3,366.000",Nonpositive profit,False,100.000
14207,14313,1,1,"1,500.000","1,200.000","-3,000.000",300.000,1,-250.000,-200.000,20.000,"-3,000.000",Nonpositive profit,False,100.000
17955,18061,1,1,"1,500.000","1,440.000","-2,592.000",60.000,1,-180.000,-172.800,4.000,"-2,592.000",Nonpositive profit,False,100.000
14007,14113,1,1,"1,500.000","1,500.000","-2,550.000",0.000,1,-170.000,-170.000,0.000,"-2,550.000",Nonpositive profit,False,100.000
13984,14090,1,1,"1,500.000","1,425.000","-2,351.250",75.000,1,-165.000,-156.750,5.000,"-2,351.250",Nonpositive profit,False,100.000
14292,14398,1,1,"1,500.000","1,455.000","-2,328.000",45.000,1,-160.000,-155.200,3.000,"-2,328.000",Nonpositive profit,False,100.000
14200,14306,1,1,"1,500.000","1,425.000","-2,280.000",75.000,1,-160.000,-152.000,5.000,"-2,280.000",Nonpositive profit,False,100.000
14130,14236,1,1,"1,500.000","1,455.000","-2,255.250",45.000,1,-155.000,-150.350,3.000,"-2,255.250",Nonpositive profit,False,100.000


,customers,net_revenue,profit,margin_pct
value_tier,,,,
High value,3306,"11,416,935.960","2,715,934.390",23.789
Nonpositive profit,4122,"6,406,767.060","-1,080,299.720",-16.862
Positive core,13224,"15,230,699.020","2,331,268.300",15.306


### Cell 14 — Measure customer profit concentration

In [14]:
positive = customers[customers["profit"] > 0].copy()
top_n = max(1, int(np.ceil(len(positive) * .20)))
positive_pool = positive["profit"].sum()
share = 100 * positive.head(top_n)["profit"].sum() / positive_pool
print(f"Top 20% of positive-profit customers produce {share:.2f}% of the positive-profit pool.")
print("Loss-making customer count:", (customers["profit"] < 0).sum())
fig, ax = plt.subplots(figsize=(9, 5))
x = 100 * np.arange(1, len(positive) + 1) / len(positive)
y = 100 * positive["profit"].cumsum() / positive_pool
ax.plot(x, y)
ax.set(xlabel="Positive-profit customers, ranked (%)", ylabel="Cumulative positive profit (%)", title="Customer profit concentration")
fig.tight_layout()
fig.savefig(OUT / "customer_concentration.png", dpi=160)
plt.show()

Top 20% of positive-profit customers produce 53.81% of the positive-profit pool.
Loss-making customer count: 4069


### Cell 15 — Compare customer segments

In [15]:
segments = summarize(df, ["Customer Segment"])
display(segments)
segments.to_csv(OUT / "segment_profitability.csv", index=False)
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=segments, x="Customer Segment", y="margin_pct", ax=ax)
ax.set_title("Weighted margin by customer segment")
plt.show()

,Customer Segment,rows,customers,gross_sales,net_revenue,profit,discounts,units,margin_pct,profit_to_gross_pct,effective_discount_pct
0,Consumer,93504,10695,"19,095,789.790","17,164,278.190","2,073,487.670","1,931,535.700",199234,12.080,10.858,10.115
1,Corporate,54789,6239,"11,168,406.630","10,030,649.800","1,202,574.960","1,137,770.700",116560,11.989,10.768,10.187
2,Home Office,32226,3718,"6,520,537.890","5,859,474.050","690,840.340","661,072.000",68285,11.790,10.595,10.138


### Cell 16 — Compare observed discount bands and no-discount records

In [16]:
discount_bands = summarize(df, ["discount_band"]).sort_values("discount_band")
display(discount_bands)
discount_bands.to_csv(OUT / "discount_bands.csv", index=False)
discount_comparison = summarize(df.assign(discount_group=np.where(df["discount_amount"] > 0, "Discounted", "No discount")), ["discount_group"])
display(discount_comparison)
print("Groups have different customer/product mix: observational comparison, not causal treatment effect.")

,discount_band,rows,customers,gross_sales,net_revenue,profit,discounts,units,margin_pct,profit_to_gross_pct,effective_discount_pct
0,0%,10028,6822,"2,042,369.140","2,042,369.140","267,412.400",0.000,21367,13.093,13.093,0.000
1,>0–5%,50143,13827,"10,215,745.610","9,909,225.720","1,171,682.080","306,523.060",106676,11.824,11.469,3.000
2,>5–10%,40116,13043,"8,173,852.120","7,530,117.510","926,511.630","643,742.600",85335,12.304,11.335,7.876
3,>10–15%,30087,11781,"6,131,787.370","5,314,179.640","625,339.120","817,610.240",64019,11.767,10.198,13.334
4,>15–20%,40116,12987,"8,176,657.730","6,725,259.470","784,061.020","1,451,398.930",85327,11.658,9.589,17.751
5,>20%,10029,6823,"2,044,322.340","1,533,250.560","191,896.720","511,103.570",21355,12.516,9.387,25.001


,discount_group,rows,customers,gross_sales,net_revenue,profit,discounts,units,margin_pct,profit_to_gross_pct,effective_discount_pct
0,Discounted,170491,20180,"34,742,365.170","31,012,032.900","3,699,490.570","3,730,378.400",362712,11.929,10.648,10.737
1,No discount,10028,6822,"2,042,369.140","2,042,369.140","267,412.400",0.000,21367,13.093,13.093,0.000


Groups have different customer/product mix: observational comparison, not causal treatment effect.


### Cell 17 — Inspect associations and candidate erosion thresholds

In [17]:
correlations = df[["discount_rate", "margin_pct", "Order Item Profit Ratio"]].corr(method="spearman")
display(correlations)
rate_summary = summarize(df, ["discount_rate"]).sort_values("discount_rate")
rate_summary["margin_change_from_prior_pp"] = rate_summary["margin_pct"].diff()
display(rate_summary)
negative_bands = discount_bands[discount_bands["profit"] < 0]
print("Observed loss-making bands (if any):")
display(negative_bands)
print("No universally causal discount threshold can be inferred from this cross-sectional dataset.")
rate_summary.to_csv(OUT / "discount_rate_diagnostics.csv", index=False)

,discount_rate,margin_pct,Order Item Profit Ratio
discount_rate,1.000,-0.002,-0.002
margin_pct,-0.002,1.000,1.000
Order Item Profit Ratio,-0.002,1.000,1.000


,discount_rate,rows,customers,gross_sales,net_revenue,profit,discounts,units,margin_pct,profit_to_gross_pct,effective_discount_pct,margin_change_from_prior_pp
0,0.000,10028,6822,"2,042,369.140","2,042,369.140","267,412.400",0.000,21367,13.093,13.093,0.000,NaN
1,0.010,10028,6818,"2,042,793.010","2,022,362.740","238,653.960","20,430.270",21329,11.801,11.683,1.000,-1.292
2,0.020,10028,6828,"2,042,818.110","2,001,957.570","237,401.160","40,861.200",21339,11.858,11.621,2.000,0.058
3,0.030,10029,6835,"2,043,343.780","1,982,037.320","222,475.430","61,306.460",21329,11.225,10.888,3.000,-0.634
4,0.040,10029,6797,"2,043,377.860","1,961,634.150","234,683.300","81,743.710",21336,11.964,11.485,4.000,0.739
5,0.050,10029,6864,"2,043,412.850","1,941,233.940","238,468.230","102,181.420",21343,12.284,11.670,5.001,0.321
6,0.060,10029,6840,"2,043,421.400","1,931,024.190","241,415.970","112,397.570",21333,12.502,11.814,5.500,0.218
7,0.070,10029,6911,"2,043,451.850","1,900,399.010","233,404.920","143,052.840",21343,12.282,11.422,7.001,-0.220
8,0.090,10029,6791,"2,043,478.390","1,859,552.080","229,769.420","183,926.310",21321,12.356,11.244,9.001,0.074
9,0.100,10029,6818,"2,043,500.480","1,839,142.230","221,921.320","204,365.880",21338,12.067,10.860,10.001,-0.290


Observed loss-making bands (if any):


,discount_band,rows,customers,gross_sales,net_revenue,profit,discounts,units,margin_pct,profit_to_gross_pct,effective_discount_pct


No universally causal discount threshold can be inferred from this cross-sectional dataset.


### Cell 18 — Examine discount patterns within product/market groups

In [18]:
within_group = summarize(df, ["Product Name", "Market", "discount_band"])
within_group = within_group[within_group["rows"] >= 30]
display(within_group.head(20))
print("Minimum 30 records per displayed cell; stratification reduces but does not eliminate confounding.")
within_group.to_csv(OUT / "within_product_market_discounts.csv", index=False)

,Product Name,Market,discount_band,rows,customers,gross_sales,net_revenue,profit,discounts,units,margin_pct,profit_to_gross_pct,effective_discount_pct
423,Field & Stream Sportsman 16 Gun Fire Safe,LATAM,>0–5%,1412,1200,"564,771.760","547,839.760","70,698.250","16,932.000",1412,12.905,12.518,2.998
417,Field & Stream Sportsman 16 Gun Fire Safe,Europe,>0–5%,1319,1147,"527,573.620","511,773.620","56,523.780","15,800.000",1319,11.045,10.714,2.995
418,Field & Stream Sportsman 16 Gun Fire Safe,Europe,>5–10%,1086,946,"434,378.280","400,242.280","55,556.810","34,136.000",1086,13.881,12.790,7.859
1281,Perfect Fitness Perfect Rip Deck,Europe,>0–5%,1896,1654,"340,383.260","330,116.660","51,868.630","10,266.600",5674,15.712,15.238,3.016
424,Field & Stream Sportsman 16 Gun Fire Safe,LATAM,>5–10%,1128,968,"451,177.440","415,645.440","46,678.890","35,532.000",1128,11.230,10.346,7.875
426,Field & Stream Sportsman 16 Gun Fire Safe,LATAM,>15–20%,1132,995,"452,777.360","372,405.360","44,893.670","80,372.000",1132,12.055,9.915,17.751
420,Field & Stream Sportsman 16 Gun Fire Safe,Europe,>15–20%,1073,931,"429,178.540","352,910.540","43,108.450","76,268.000",1073,12.215,10.044,17.771
1084,Nike Men's Free 5.0+ Running Shoe,LATAM,>0–5%,1010,924,"305,469.450","296,314.450","41,632.410","9,155.000",3055,14.050,13.629,2.997
425,Field & Stream Sportsman 16 Gun Fire Safe,LATAM,>10–15%,846,750,"338,383.080","293,263.080","38,241.090","45,120.000",846,13.040,11.301,13.334
1287,Perfect Fitness Perfect Rip Deck,LATAM,>0–5%,2021,1773,"364,679.210","353,733.410","37,823.020","10,945.800",6079,10.693,10.372,3.001


Minimum 30 records per displayed cell; stratification reduces but does not eliminate confounding.


### Cell 19 — Run transparent discount what-if scenarios

In [19]:
scenarios = pd.DataFrame([discount_scenario(df, r) for r in [0, .05, .10, .15, .20, .25]])
display(scenarios)
scenarios.to_csv(OUT / "discount_scenarios.csv", index=False)
# Each record's fixed-cost break-even discount = reported profit + existing discount, divided by gross sales.
df["modeled_break_even_discount"] = (df["profit"] + df["discount_amount"]) / df["gross_sales"]
print("Break-even discount percentiles (negative values mean a loss even without discount):")
display((100 * df["modeled_break_even_discount"]).quantile([0, .1, .25, .5, .75, .9, 1]))
print("Assumptions: unchanged gross prices, quantities, mix and implied costs; no demand response.")

,scenario_discount_pct,scenario_net_revenue,scenario_profit,profit_change,scenario_margin_pct
0,0.000,"36,784,734.310","7,697,235.240","3,730,332.270",20.925
1,5.000,"34,945,497.594","5,857,998.524","1,891,095.554",16.763
2,10.000,"33,106,260.879","4,018,761.809","51,858.839",12.139
3,15.000,"31,267,024.163","2,179,525.093","-1,787,377.877",6.971
4,20.000,"29,427,787.448","340,288.378","-3,626,614.592",1.156
5,25.000,"27,588,550.733","-1,498,948.337","-5,465,851.307",-5.433


Break-even discount percentiles (negative values mean a loss even without discount):


0.000   -275.000
0.100    -20.002
0.250     14.446
0.500     33.571
0.750     45.252
0.900     52.304
1.000     62.539
Name: modeled_break_even_discount, dtype: float64

Assumptions: unchanged gross prices, quantities, mix and implied costs; no demand response.


### Cell 20 — Diagnose markets, regions and destination countries

In [20]:
markets = summarize(df, ["Market"])
regions = summarize(df, ["Order Region"])
countries = summarize(df, ["Order Country"])
for name, table in [("market", markets), ("region", regions), ("country", countries)]:
    display(table)
    table.to_csv(OUT / f"{name}_profitability.csv", index=False)
weak_markets = markets[(markets["net_revenue"] >= markets["net_revenue"].median()) & (markets["margin_pct"] < metrics["margin_pct"])]
print("Strong-revenue/weak-profit markets: revenue >= market median and margin below portfolio margin.")
display(weak_markets)

,Market,rows,customers,gross_sales,net_revenue,profit,discounts,units,margin_pct,profit_to_gross_pct,effective_discount_pct
1,Europe,50252,11657,"10,872,396.600","9,769,198.230","1,169,442.960","1,103,210.750",105238,11.971,10.756,10.147
2,LATAM,51594,9325,"10,277,612.640","9,235,761.990","1,123,321.610","1,041,863.150",112942,12.163,10.930,10.137
3,Pacific Asia,41260,13267,"8,273,743.580","7,434,262.580","857,753.440","839,493.310",83680,11.538,10.367,10.146
4,USCA,25799,6256,"5,066,528.610","4,553,499.870","564,313.780","513,034.820",56616,12.393,11.138,10.126
0,Africa,11614,3311,"2,294,452.880","2,061,679.370","252,071.180","232,776.370",25603,12.226,10.986,10.145


,Order Region,rows,customers,gross_sales,net_revenue,profit,discounts,units,margin_pct,profit_to_gross_pct,effective_discount_pct
22,Western Europe,27109,7695,"5,894,380.660","5,296,002.840","625,446.080","598,384.310",56504,11.810,10.611,10.152
3,Central America,28341,6564,"5,665,711.990","5,093,849.660","616,341.570","571,869.410",62091,12.100,10.878,10.094
12,South America,14935,4131,"2,960,881.350","2,660,243.660","335,154.400","300,641.190",32647,12.599,11.319,10.154
10,Northern Europe,9792,3340,"2,155,830.610","1,939,362.410","233,450.600","216,470.730",20402,12.037,10.829,10.041
17,Southern Europe,9431,3192,"2,047,918.780","1,837,525.800","230,829.230","210,395.350",19526,12.562,11.271,10.274
15,Southeast Asia,9539,4101,"1,932,495.530","1,738,553.420","211,342.820","193,945.360",18835,12.156,10.936,10.036
11,Oceania,10148,4042,"2,016,654.160","1,809,996.530","201,478.020","206,660.620",20430,11.131,9.991,10.248
1,Caribbean,8318,2506,"1,651,019.300","1,481,668.670","171,825.640","169,352.550",18204,11.597,10.407,10.257
13,South Asia,7731,3153,"1,553,680.890","1,397,364.560","165,703.900","156,318.690",15624,11.858,10.665,10.061
21,West of USA,7993,2400,"1,571,415.930","1,412,253.580","164,940.660","159,164.320",17547,11.679,10.496,10.129


,Order Country,rows,customers,gross_sales,net_revenue,profit,discounts,units,margin_pct,profit_to_gross_pct,effective_discount_pct
48,Estados Unidos,24840,6102,"4,879,667.570","4,385,242.450","540,413.070","494,431.000",54433,12.323,11.075,10.132
53,Francia,13222,4245,"2,879,942.310","2,590,323.470","327,828.580","289,622.010",27676,12.656,11.383,10.057
102,México,13172,3706,"2,633,195.240","2,368,426.060","303,278.370","264,772.550",28794,12.805,11.518,10.055
2,Alemania,9564,3188,"2,074,171.780","1,862,577.570","194,827.080","211,596.410",19803,10.460,9.393,10.201
20,Brasil,7987,2399,"1,594,319.920","1,433,100.980","186,713.640","161,220.720",17405,13.029,11.711,10.112
...,...,...,...,...,...,...,...,...,...,...,...
98,Mongolia,79,28,"15,481.740","14,059.500",-286.230,"1,422.260",185,-2.036,-1.849,9.187
18,Bosnia y Herzegovina,45,16,"8,869.090","7,804.970",-327.380,"1,064.160",83,-4.195,-3.691,11.999
127,Ruanda,86,30,"18,913.020","17,101.560",-330.020,"1,811.480",185,-1.930,-1.745,9.578
146,Taiwán,29,13,"6,396.330","5,821.110","-1,374.540",575.220,52,-23.613,-21.490,8.993


Strong-revenue/weak-profit markets: revenue >= market median and margin below portfolio margin.


,Market,rows,customers,gross_sales,net_revenue,profit,discounts,units,margin_pct,profit_to_gross_pct,effective_discount_pct
1,Europe,50252,11657,"10,872,396.600","9,769,198.230","1,169,442.960","1,103,210.750",105238,11.971,10.756,10.147
3,Pacific Asia,41260,13267,"8,273,743.580","7,434,262.580","857,753.440","839,493.310",83680,11.538,10.367,10.146


### Cell 21 — Generate research paper and executive summary

In [21]:
from report_builder import write_reports
write_reports(df, audit, rejected, OUT)
print("Research paper and executive summary written in outputs, in Markdown and Word formats.")

Verified figures: {'rows': 180519.0, 'customers': 20652.0, 'gross_sales': 36784734.31, 'net_revenue': 33054402.04, 'profit': 3966902.97, 'discounts': 3730378.4, 'margin_pct': 12.0011, 'profit_to_gross_pct': 10.7841, 'discount_impact_ratio_pct': 48.4636, 'modeled_margin_erosion_pp': 8.9241}
Loss customers: 4069 Loss products: 3 Loss categories: 0
Research paper and executive summary written in outputs, in Markdown and Word formats.


### Cell 22 — Check dashboard source and run it locally

In [22]:
import subprocess
import sys
subprocess.run([sys.executable, "-m", "py_compile", "app.py", "analysis_core.py"], check=True)
print("Dashboard syntax check passed.")
print("In a terminal in this directory, run: streamlit run app.py")
# Do not launch a long-running server inside this notebook cell.
# The app supports upload, segment/category/product/market/region/status filters,
# an observed-discount slider, customer rankings, product plots, heatmaps, and scenarios.

Dashboard syntax check passed.
In a terminal in this directory, run: streamlit run app.py


## Submission and interpretation checklist
- Include the executed notebook, research paper, executive summary, dashboard source and requirements.
- To host: put the project in a repository, create a Streamlit-hosted app pointing to `app.py`, and upload the CSV through the application. Do not commit private customer data to a public repository.
- A local app is provided; no public deployment or live URL is claimed.
- Financial aliases are not double counted; losses are retained; invalid records are quarantined.
- No unsupported dollar currency, time trend, unique order counts, shipping cost attribution, or causal discount conclusions.
- Use aggregate results for government stakeholders; keep customer IDs and raw data access restricted.
- Validate accounting status, cost completeness, currency, licensing/provenance and recognition rules before real decisions.